In [1]:
from dotenv import load_dotenv
load_dotenv()  # This automatically injects your .env file into Python's environment
# Needs a .env file (repo root or notebook/) with: GEMINI_API_KEY=your-key-here
# .env is gitignored -- never commit it.

True

In [ ]:
import json
import os
import sys
import time
from pathlib import Path

from google import genai
from google.genai import types
from google.genai.errors import ClientError, ServerError
from intelli_e_reader.llm_baseline import build_batch_request, parse_batch_results, write_batch_jsonl_files
from intelli_e_reader.data_utils import build_family1_word_cefr, build_efllex_cefr, extend_family1_with_efllex

import pandas as pd

sys.path.insert(0, os.path.abspath(".."))  # repo root, so `config`/`intelli_e_reader` import cleanly

PROMPT_DIR = Path("../prompts/cefr_classification")
GEMINI_MODEL = "gemini-3.6-flash"  # cheap/fast tier -- verify this model ID is still current in AI Studio

client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])

In [17]:
system_prompt = (PROMPT_DIR / "system_prompt.md").read_text()
few_shot = json.loads((PROMPT_DIR / "few_shot_examples.json").read_text())

RESPONSE_SCHEMA = {
    "type": "OBJECT",
    "properties": {
        "cefr_int": {"type": "INTEGER"},
        "cefr_level": {"type": "STRING", "enum": ["A1", "A2", "B1", "B2", "C1", "C2"]},
        "confidence": {"type": "STRING", "enum": ["high", "medium", "low"]},
        "reasoning": {"type": "STRING"},
    },
    "required": ["cefr_int", "cefr_level", "confidence", "reasoning"],
}


def word_target_text(word, pos):
    return f'Word: "{word}"\nPart of speech: {pos}'


# Few-shot examples as alternating user/model turns (not a text blob) -- see prompts/cefr_classification/README.md
few_shot_contents = []
for ex in few_shot:
    few_shot_contents.append(
        types.Content(role="user", parts=[types.Part.from_text(text=word_target_text(ex["word"], ex["pos"]))])
    )
    assistant_json = json.dumps({k: ex[k] for k in ("cefr_int", "cefr_level", "confidence", "reasoning")})
    few_shot_contents.append(types.Content(role="model", parts=[types.Part.from_text(text=assistant_json)]))

In [18]:
family1 = build_family1_word_cefr()
extended = extend_family1_with_efllex(family1, build_efllex_cefr())

# exclude words already used as few-shot calibration examples, so the test is on genuinely unseen words
few_shot_words = {(ex["word"], ex["pos"]) for ex in few_shot}
candidates = extended[~extended.set_index(["word", "pos"]).index.isin(few_shot_words)]
test_sample = candidates.sample(5, random_state=42)[["word", "pos", "cefr", "cefr_int"]].reset_index(drop=True)
test_sample

,word,pos,cefr,cefr_int
0,accommodate,VERB,B,3
1,atomique,NOUN,B,4
2,university,NOUN,A,1
3,again,NOUN,B,4
4,single-handed,ADJ,B,4


In [ ]:



def generate_with_retry(contents, config, max_retries=4, base_delay=5.0):
    """Retry on transient 503 (model overloaded) with exponential backoff.
    Doesn't retry on other errors (e.g. 400s) -- those are real bugs, not transient."""
    for attempt in range(max_retries):
        try:
            return client.models.generate_content(model=GEMINI_MODEL, contents=contents, config=config)
        except ServerError as e:
            if getattr(e, "code", None) != 503 or attempt == max_retries - 1:
                raise
            delay = base_delay * (2 ** attempt)
            print(f"  503 (overloaded), retrying in {delay:.0f}s ({attempt+1}/{max_retries})...")
            time.sleep(delay)


results = []
total_prompt_tokens = 0
total_output_tokens = 0

for _, row in test_sample.iterrows():
    target_content = types.Content(
        role="user", parts=[types.Part.from_text(text=word_target_text(row["word"], row["pos"]))]
    )
    response = generate_with_retry(
        contents=few_shot_contents + [target_content],
        config=types.GenerateContentConfig(
            system_instruction=system_prompt,
            response_mime_type="application/json",
            response_schema=RESPONSE_SCHEMA,
        ),
    )
    parsed = json.loads(response.text)
    parsed["word"] = row["word"]
    parsed["pos"] = row["pos"]
    parsed["our_cefr"] = row["cefr"]
    parsed["our_cefr_int"] = row["cefr_int"]
    results.append(parsed)

    usage = response.usage_metadata
    total_prompt_tokens += usage.prompt_token_count
    total_output_tokens += usage.candidates_token_count
    time.sleep(1)  # gentle on free-tier rate limits

results_df = pd.DataFrame(results)
print(results_df[["word", "pos", "our_cefr", "cefr_level", "confidence", "reasoning"]].to_string(index=False))
print()

total_tokens = total_prompt_tokens + total_output_tokens
per_word = total_tokens / len(test_sample)
print(f"prompt tokens: {total_prompt_tokens}, output tokens: {total_output_tokens}, total: {total_tokens}")
print(f"avg per word: {per_word:.0f} tokens")
print(f"projected total for all {len(extended):,} words: {per_word * len(extended):,.0f} tokens")
print("Check the current per-token price for this model/tier on Google AI Studio's pricing page, "
      "then multiply the projected token count by that rate -- no verified-current Gemini price to hardcode here.")

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


         word  pos our_cefr cefr_level confidence                          reasoning
  accommodate VERB        B         B2       high                 common formal verb
     atomique NOUN        B         C2       high rare foreign loanword, specialized
   university NOUN        A         A1       high        high-frequency, basic topic
        again NOUN        B         C2       high          extremely rare noun usage
single-handed  ADJ        B         C1       high      compound adjective, idiomatic

prompt tokens: 2125, output tokens: 179, total: 2304
avg per word: 461 tokens
projected total for all 18,585 words: 8,563,968 tokens
Check the current per-token price for this model/tier on Google AI Studio's pricing page, then multiply the projected token count by that rate -- no verified-current Gemini price to hardcode here.


In [ ]:
BATCH_DIR = Path("../batch_jobs/cefr_llm_baseline")

# idempotent -- safe to re-run on a fresh kernel, regenerates the same files
batch_targets = candidates[["word", "pos"]].reset_index(drop=True)
batch_paths = write_batch_jsonl_files(batch_targets, few_shot_contents, system_prompt, BATCH_DIR, chunk_size=3000)
print(f"{len(batch_paths)} files covering {len(batch_targets):,} words:")
for p in batch_paths:
    print(" ", p, f"({p.stat().st_size / 1024:.0f} KB)")

In [ ]:
# validate on a tiny job first, before trusting the real ~7-file submission
validation_path = BATCH_DIR / "validation_5row.jsonl"
with open(validation_path, "w") as f:
    for _, row in test_sample.iterrows():
        line = build_batch_request(row["word"], row["pos"], few_shot_contents, system_prompt)
        f.write(json.dumps(line) + "\n")

uploaded = client.files.upload(file=str(validation_path), config=types.UploadFileConfig(mime_type="jsonl"))
validation_job = client.batches.create(
    model=GEMINI_MODEL, src=uploaded.name, config={"display_name": "cefr-validation-5row"}
)
print(validation_job.name, validation_job.state.name)

In [14]:
# Poll until the validation job finishes -- re-run this cell if it's still RUNNING/PENDING.
validation_job = client.batches.get(name=validation_job.name)
print(validation_job.state.name)

if validation_job.state.name == "JOB_STATE_SUCCEEDED":
    content = client.files.download(file=validation_job.dest.file_name)
    validation_results = parse_batch_results(content)
    print(pd.DataFrame(validation_results))
elif validation_job.state.name == "JOB_STATE_FAILED":
    print(validation_job.error)

JOB_STATE_SUCCEEDED
            word   pos  cefr_int cefr_level confidence  \
0    accommodate  VERB         4         B2       high   
1       atomique  NOUN         6         C2        low   
2     university  NOUN         2         A2       high   
3          again  NOUN         6         C2       high   
4  single-handed   ADJ         5         C1       high   

                           reasoning  
0                 common formal verb  
1  rare foreignism, specialized noun  
2    common everyday noun, education  
3       extremely rare nominal usage  
4       compound idiomatic adjective  


## Only run the cells below once the validation job above shows `JOB_STATE_SUCCEEDED`
## with sane-looking output. This submits the real ~18.5K-word run.

In [ ]:
job_names_path = BATCH_DIR / "job_names.json"
job_names = json.loads(job_names_path.read_text()) if job_names_path.exists() else {}

for path in batch_paths:
    if path.name in job_names:
        print(f"already submitted: {path.name} -> {job_names[path.name]}")
        continue

    for attempt in range(6):
        try:
            uploaded = client.files.upload(file=str(path), config=types.UploadFileConfig(mime_type="jsonl"))
            job = client.batches.create(model=GEMINI_MODEL, src=uploaded.name, config={"display_name": path.stem})
            job_names[path.name] = job.name
            job_names_path.write_text(json.dumps(job_names, indent=2))  # save after every success, not just at the end
            print(job.name, job.state.name, "<-", path.name)
            break
        except ClientError as e:
            if getattr(e, "code", None) != 429 or attempt == 5:
                raise
            delay = 15 * (2 ** attempt)
            print(f"  429 (quota), retrying {path.name} in {delay}s ({attempt+1}/6)...")
            time.sleep(delay)
    time.sleep(10)  # spacing between submissions, to stay under rate limits

print(f"\n{len(job_names)}/{len(batch_paths)} files submitted, saved to {job_names_path}")

In [39]:
# Re-run this cell any time to check progress -- safe to re-run repeatedly, including
# after a kernel restart (reloads job names from disk instead of relying on batch_jobs).
job_names_path = BATCH_DIR / "job_names.json"
job_names = json.loads(job_names_path.read_text())
statuses = {fname: client.batches.get(name=name) for fname, name in job_names.items()}
for fname, job in statuses.items():
    print(fname, "->", job.name, job.state.name)

done = all(j.state.name in ("JOB_STATE_SUCCEEDED", "JOB_STATE_FAILED") for j in statuses.values())
print(f"\n{len(job_names)}/7 files submitted so far.",
      "all submitted jobs finished:" if done else "still running, check back later:", done)

cefr_batch_000.jsonl -> batches/jrd0ifdln5muxsozb1jez8wy4ijg9az19437 JOB_STATE_SUCCEEDED
cefr_batch_001.jsonl -> batches/k6yd49wt16gczlj83fz2rf6ytstpueofi76c JOB_STATE_SUCCEEDED
cefr_batch_002.jsonl -> batches/zbgtp58hfrzag7qj2oytqbzcrlurlkodebkd JOB_STATE_SUCCEEDED
cefr_batch_003.jsonl -> batches/j1db26of6m2oe9peihs0p4e9nji1e2qqklne JOB_STATE_SUCCEEDED
cefr_batch_004.jsonl -> batches/rop01aehe8jgqocqmfg00ul2zfc9r5h7vlt4 JOB_STATE_SUCCEEDED
cefr_batch_005.jsonl -> batches/43ksjspz04c2hy0mfmy4p3f8kzhfixqn1wej JOB_STATE_SUCCEEDED
cefr_batch_006.jsonl -> batches/lwlanydsko43byj9ahqkcdmq30echtqevoco JOB_STATE_SUCCEEDED

7/7 files submitted so far. all submitted jobs finished: True


In [ ]:
# once all jobs show JOB_STATE_SUCCEEDED: download each chunk's raw results to disk
all_results = []
RESULTS = Path("../batch_jobs/cefr_llm_baseline/results")
for fname, job in statuses.items():
    print(fname)
    if job.state.name != "JOB_STATE_SUCCEEDED":
        print(f"skipping {fname}, state={job.state.name}" + (f" error={job.error}" if job.error else ""))
        continue
    content = client.files.download(file=job.dest.file_name)
    local_filename = f"{RESULTS}/{fname}"
    with open(local_filename, "wb") as f:
        f.write(content)
    print(local_filename)

In [9]:
all_results = []

RESULTS = Path("../batch_jobs/cefr_llm_baseline/results")

results_files = os.listdir(RESULTS)
for files in results_files:
    file_name = f"{RESULTS}/{files}"
    with open(file_name, "r") as f:
        print(file_name)
        data = parse_batch_results(f)
        all_results.extend(data)

../batch_jobs/cefr_llm_baseline/results/cefr_batch_005.jsonl
../batch_jobs/cefr_llm_baseline/results/cefr_batch_006.jsonl
../batch_jobs/cefr_llm_baseline/results/cefr_batch_001.jsonl
Error in line 112! error : Expecting value: line 1 column 1 (char 0)
Error in line 602! error : Expecting value: line 1 column 1 (char 0)
Error in line 755! error : Expecting value: line 1 column 1 (char 0)
Error in line 2106! error : Expecting value: line 1 column 1 (char 0)
Error in line 2302! error : Expecting value: line 1 column 1 (char 0)
Error in line 2604! error : Expecting value: line 1 column 1 (char 0)
Error in line 2968! error : Expecting value: line 1 column 1 (char 0)
../batch_jobs/cefr_llm_baseline/results/cefr_batch_002.jsonl
Error in line 1249! error : Expecting value: line 1 column 1 (char 0)
Error in line 1343! error : Expecting value: line 1 column 1 (char 0)
../batch_jobs/cefr_llm_baseline/results/cefr_batch_004.jsonl
../batch_jobs/cefr_llm_baseline/results/cefr_batch_003.jsonl
../batc

In [10]:
all_results[0]

{'word': 'splashing',
 'pos': 'NOUN',
 'cefr_int': 4,
 'cefr_level': 'B2',
 'confidence': 'medium',
 'reasoning': 'descriptive action noun'}

In [11]:
llm_baseline_df = pd.DataFrame(all_results,)
llm_baseline_df.to_parquet(RESULTS / "llm_baseline_results.parquet", index=False)
print(f"{len(llm_baseline_df):,} predictions -> {RESULTS / 'llm_baseline_results.parquet'}")

18,571 predictions -> ../batch_jobs/cefr_llm_baseline/results/llm_baseline_results.parquet


In [13]:
print(llm_baseline_df.shape)
llm_baseline_df.head()

(18571, 6)


,word,pos,cefr_int,cefr_level,confidence,reasoning
0,splashing,NOUN,4,B2,medium,descriptive action noun
1,split_up,VERB,3,B1,high,common phrasal verb
2,splurge,VERB,5,C1,high,"informal, specialized vocabulary"
3,spoilt,ADJ,4,B2,high,common descriptive adjective
4,spontaneity,NOUN,5,C1,high,"abstract noun, low-frequency"


In [22]:
merged = llm_baseline_df.merge(extended[["word", "pos", "cefr", "cefr_int"]], on=["word", "pos"], how="left",suffixes=["_llm",""])
print(f"exact match with our lookup-table cefr: {(merged['cefr'] == merged['cefr_level'].str[0]).mean():.1%}")

exact match with our lookup-table cefr: 52.1%


In [23]:
merged.shape
merged.head()

,word,pos,cefr_int_llm,cefr_level,confidence,reasoning,cefr,cefr_int
0,splashing,NOUN,4,B2,medium,descriptive action noun,A,2
1,split_up,VERB,3,B1,high,common phrasal verb,B,4
2,splurge,VERB,5,C1,high,"informal, specialized vocabulary",C,5
3,spoilt,ADJ,4,B2,high,common descriptive adjective,C,5
4,spontaneity,NOUN,5,C1,high,"abstract noun, low-frequency",C,5
